# Vehicle dataset check (no training)
Downloads the Roboflow "Classification of Vehicles" dataset (version 9) and checks it: folders, split sizes, boxes per class, and pictures with their boxes drawn.

## a) Download the dataset
Installs the Roboflow library, reads your API key from Kaggle's secret store (it is never printed), and downloads version 9 in YOLO format to `/kaggle/working/vehicles_v9`.

In [ ]:
!pip install -q roboflow

from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")  # never printed

rf = Roboflow(api_key=api_key)
project = rf.workspace("labelme-to-darknet-yolo").project("classification-of-vehicles")
dataset = project.version(9).download("yolov11", location="/kaggle/working/vehicles_v9")
print("Downloaded to:", dataset.location)

## b) Folder tree and data.yaml
Shows how the download is organised (2 levels deep, with a file count for each folder) and prints `data.yaml`, which lists the class names.

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/vehicles_v9")


# Print folders and files up to 2 levels deep. Folders show a file count instead of every file.
def show_tree(folder, depth=0, max_depth=2):
    for item in sorted(folder.iterdir()):
        if item.is_dir():
            count = sum(1 for f in item.rglob("*") if f.is_file())
            print("    " * depth + f"{item.name}/  ({count} files)")
            if depth + 1 < max_depth:
                show_tree(item, depth + 1, max_depth)
        else:
            print("    " * depth + item.name)


show_tree(ROOT)
print("\n--- data.yaml ---")
print((ROOT / "data.yaml").read_text())

## c) Images and labels per split
Counts images and label files in train, valid and test, and the share of images in each. Every image should have one label file.

In [ ]:
import pandas as pd

IMAGE_TYPES = (".jpg", ".jpeg", ".png")

rows = []
for split in ("train", "valid", "test"):
    images = [p for p in (ROOT / split / "images").glob("*") if p.suffix.lower() in IMAGE_TYPES]
    labels = list((ROOT / split / "labels").glob("*.txt"))
    rows.append({"split": split, "images": len(images), "labels": len(labels)})

splits = pd.DataFrame(rows)
splits["percent of images"] = (100 * splits["images"] / splits["images"].sum()).round(1)
print(splits.to_string(index=False))
print("Total images:", splits["images"].sum())

## d) Boxes per class
Counts how many boxes each class has in each split. Classes with very few boxes are hard to learn. Saved as `class_counts.csv`.

In [ ]:
import yaml
from collections import Counter

names = yaml.safe_load((ROOT / "data.yaml").read_text())["names"]
if isinstance(names, dict):  # some files store names as {0: "Bike", ...}
    names = [names[k] for k in sorted(names)]

counts = {split: Counter() for split in ("train", "valid", "test")}
for split in counts:
    for label in (ROOT / split / "labels").glob("*.txt"):
        for line in label.read_text().splitlines():
            if line.strip():
                counts[split][int(line.split()[0])] += 1

table = pd.DataFrame({split: [counts[split][i] for i in range(len(names))] for split in counts},
                     index=names)
table["total"] = table.sum(axis=1)
table.index.name = "class"
print(table)
table.to_csv("/kaggle/working/class_counts.csv")

## e) One label file and its picture
Prints 3 lines of one label file (class number, then box centre x, centre y, width, height, from 0 to 1) and draws those boxes with class names. Saved as `one_label.jpg`.

In [ ]:
from PIL import Image, ImageDraw


# Draw every box in a label file on its image, with the class name above each box.
def draw_boxes(image_path, label_path):
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    for line in label_path.read_text().splitlines():
        values = line.split()
        if not values:
            continue
        class_id, numbers = int(values[0]), list(map(float, values[1:]))
        if len(numbers) == 4:  # a normal box
            cx, cy, w, h = numbers
            box = [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
        else:                  # a polygon: use its outer edges
            xs, ys = numbers[0::2], numbers[1::2]
            box = [min(xs), min(ys), max(xs), max(ys)]
        left, top = box[0] * image.width, box[1] * image.height
        draw.rectangle([left, top, box[2] * image.width, box[3] * image.height],
                       outline="red", width=3)
        draw.text((left + 3, max(0, top - 12)), names[class_id], fill="yellow")
    return image


# Find the image that belongs to a label file (same name, in the images folder).
def image_for(label_path):
    for suffix in IMAGE_TYPES:
        candidate = label_path.parent.parent / "images" / (label_path.stem + suffix)
        if candidate.exists():
            return candidate


label = next(l for l in sorted((ROOT / "train" / "labels").glob("*.txt")) if l.read_text().strip())
print("Label file:", label.name)
print("\n".join(label.read_text().splitlines()[:3]))

example = draw_boxes(image_for(label), label)
example.save("/kaggle/working/one_label.jpg")
display(example)

## f) 3x3 grid of random training images
Nine random training pictures with their boxes, to spot wrong or missing labels quickly. The fixed seed (42) picks the same nine every run. Saved as `grid.jpg`.

In [ ]:
import random

random.seed(42)
TILE = 400

with_boxes = [l for l in sorted((ROOT / "train" / "labels").glob("*.txt")) if l.read_text().strip()]
grid = Image.new("RGB", (3 * TILE, 3 * TILE), "white")
for i, label in enumerate(random.sample(with_boxes, 9)):
    tile = draw_boxes(image_for(label), label).resize((TILE, TILE))
    grid.paste(tile, ((i % 3) * TILE, (i // 3) * TILE))

grid.save("/kaggle/working/grid.jpg")
display(grid)

## g) Delete the downloaded dataset
The dataset is large, and anything left in `/kaggle/working` becomes notebook output. Deleting it keeps the download small: only the CSV and the two pictures remain.

In [ ]:
import shutil

shutil.rmtree(ROOT)
print("Deleted", ROOT)
print("Kept:", sorted(p.name for p in Path("/kaggle/working").iterdir()))